# SEC Insight — 10-K Analysis Demo

This notebook walks through using `sec-insight` to:
1. Fetch a 10-K filing from SEC EDGAR
2. Extract the Risk Factors section
3. Summarize financial highlights from MD&A
4. Compare summaries across multiple tickers

Model: `facebook/bart-large-cnn` via Hugging Face `transformers`

In [ ]:
# Install dependencies if running in Colab
# !pip install transformers torch requests

In [ ]:
import sys
sys.path.insert(0, '..')

from src.fetcher import EDGARFetcher
from src.analyzer import SECAnalyzer
import json

## 1. Load the analyzer (downloads model on first run)

In [ ]:
analyzer = SECAnalyzer(device=-1)  # change to device=0 for GPU
fetcher = EDGARFetcher()

## 2. Analyze a single ticker

In [ ]:
ticker = "MSFT"
filing_text = fetcher.fetch_filing_text(ticker)
print(f"Loaded {len(filing_text):,} characters for {ticker}")

In [ ]:
results = analyzer.analyze(filing_text)
results["ticker"] = ticker

print("=== RISK FACTORS ===")
print(results["risk_factors"]["summary"])
print()
print("=== FINANCIAL HIGHLIGHTS ===")
print(results["financial_highlights"]["summary"])

## 3. Compare risk factors across multiple companies

In [ ]:
tickers = ["AAPL", "MSFT", "GOOGL"]
comparison = {}

for t in tickers:
    print(f"Processing {t}...")
    text = fetcher.fetch_filing_text(t)
    if text:
        res = analyzer.analyze(text)
        comparison[t] = res["risk_factors"]["summary"]
    else:
        comparison[t] = "Fetch failed"

for ticker, summary in comparison.items():
    print(f"\n{'='*50}")
    print(f"  {ticker} — Risk Factors")
    print('='*50)
    print(summary)

## 4. Use a local filing file (useful for testing or NDA'd filings)

In [ ]:
# Uncomment and set your file path
# from pathlib import Path
# local_text = Path("../data/my_filing.txt").read_text()
# results = analyzer.analyze(local_text)
# print(json.dumps(results, indent=2))